In [ ]:
# Get the password
password = getpass.getpass("Enter your password: ")

In [ ]:
# Imported libraries.
import requests
import importlib
from bs4 import BeautifulSoup
import pandas as pd
import snowflake.connector
import sqlalchemy as db
import numpy as np
import import_ipynb
import re
from datetime import datetime, timedelta
import getpass


# Import my files.
import snowflake_functions as sf  # Import the notebook as a module.
import transform_data as td 
import extract_data as ed 
from skills_array import get_skills_array
skills_array = get_skills_array() # get full list of skills.

In [2]:
# upload latest function
importlib.reload(sf)
importlib.reload(ed)
importlib.reload(td)

# connection snowflake
engine = sf.connect_snowflake(password)
connection = engine.connect()

# intialise tables that will later be loaded to snowflake
job_table = pd.DataFrame(columns=['JOB_ID', 'TITLE','COMPANY','LOCATION','EMPLOYMENT_TYPE','SALARY','PAY_PERIOD','POST_DATE'])
job_skills_table = pd.DataFrame(columns=["skill_id", "skill_name"])


# get the job_id of the last entry in the jobs table
job_id = sf.last_job_id(connection) + 1

# go through many pages
max_pages = 2
for j in range(max_pages):
    
    # get all job URL links
    full_urls = ed.get_URLs_to_jobs(j) 
    
    # go through all the URLs and extract relevant data.
    for k, url in enumerate(full_urls):   
        
        # Scrape job URL.
        response = requests.get(url) 
        soup = BeautifulSoup(response.content, "html.parser") 
        
        # Enter job ad in to jobs table data frame
        job_props, req_skills = ed.get_job_data(soup, skills_array) # extract the information from job
        parsed_salary = td.parse_salary(job_props[4]) # clean up salary   
        job_table = td.update_job_table(job_props, parsed_salary, job_table, job_id + k) # put into data frame.


        # get that entries id, get skill ids and enter it into job_skills table.
        df_skill_ids = sf.get_skill_ids(req_skills, connection)
        job_skills_table = td.update_job_skills_table(job_props, parsed_salary, job_skills_table, df_skill_ids, job_id + k)
        

# upload tables to snowflake database
job_table.to_sql('jobs', con=engine, if_exists='append', index=False)
job_skills_table.to_sql('job_skills', con=engine, if_exists='append', index=False)

# Close connection
connection.close()

Connected to Snowflake!
Machine Learning & Data Engineer | Fundo Loans | Sydney NSW | Full time | 140000 | annually | 2025-02-25
Data Scientist | Expert360 | Sydney NSW | Full time | None | None | None
Senior Data Engineer (Contexa) | FinXL IT Professional Services | Sydney NSW | Contract/Temp | None | None | 2025-02-27
Machine Learning & Data Engineer | Fundo Loans | Sydney NSW | Full time | 140000 | annually | 2025-02-25
Data Engineer | Bluestone | Sydney NSW | Full time | None | None | 2025-02-22
Senior Data Engineer (Quantexa) | FinXL IT Professional Services | Sydney NSW | Contract/Temp | None | None | 2025-02-27
Principal Data Engineer | Attribute Group | Sydney NSW | Contract/Temp | None | None | 2025-02-18
Lead Data Engineer | Akkodis | Sydney NSW | Contract/Temp | None | None | None
Data Engineering Manager | Talenza | Sydney NSW | Full time | 200000 | annually | 2025-02-27
Data Engineer | Cuscal Limited (SR) | Sydney NSW | Full time | None | None | 2025-02-22
Senior Cloud Dat

In [ ]:
# Close connection
connection.close()